In [21]:
from get_article_links import api_get
import mwparserfromhell
import regex

In [22]:
data = api_get("List of current heads of state and government")

In [23]:
def retrieve_wikitext(data) -> str:
    pages = data["query"]["pages"]

    page_id = list(pages.keys())[0]

    return pages[page_id]["revisions"][0]["slots"]["main"]["*"]

wikitext = retrieve_wikitext(data)

In [24]:
def parser(wikitext):
    parsed = mwparserfromhell.parse(wikitext)

    # 525 is the last president (zimbabwe)
    # 13 is the first flag
    # this includes a lot of junk data (flags, languages)
    table = parsed.filter_templates()[13:595]

    # remove flags and languages
    table = [entry.strip("{{}}") for entry in table]

    # get only people from table
    people = [regex.findall(r"\[\[(.*?)\]\]", entry) for entry in table]

    # drop empty entries
    return [entry for entry in people if entry]

world_leaders = parser(wikitext)

In [25]:
world_leaders

[['Supreme Leader of Afghanistan|Supreme Leader', 'Hibatullah Akhundzada'],
 ['Prime Minister of Albania|Prime Minister', 'Edi Rama'],
 ['President of Algeria|President', 'Abdelmadjid Tebboune'],
 ["List of representatives of the co-princes of Andorra|Co-Prince's Representative",
  'Eduard Ibáñez'],
 ["List of representatives of the co-princes of Andorra|Co-Prince's Representative",
  'Patrice Faure'],
 ['Prime Minister of Andorra|Prime Minister',
  'Xavier Espot Zamora|Xavier Espot'],
 ['President of Angola|President', 'João Lourenço'],
 ['Governor-General of Antigua and Barbuda|Governor-General',
  'Rodney Williams'],
 ['Prime Minister of Antigua and Barbuda|Prime Minister', 'Gaston Browne'],
 ['President of Argentina|President', 'Javier Milei'],
 ['Prime Minister of Armenia|Prime Minister', 'Nikol Pashinyan'],
 ['Governor-General of Australia|Governor-General', 'Sam Mostyn'],
 ['Prime Minister of Australia|Prime Minister', 'Anthony Albanese'],
 ['Chancellor of Austria|Chancellor', '

In [33]:
import subprocess
subprocess.run(["uv", "pip", "install", "rdflib"], check=True)

CompletedProcess(args=['uv', 'pip', 'install', 'rdflib'], returncode=0)

In [34]:
import re
from rdflib import Graph, Literal, Namespace
from rdflib.namespace import RDF, RDFS, OWL

In [52]:
# NOISE: strings that look like people but aren't
PERSON_BLACKLIST = {
    'BBC News', 'Al Jazeera', 'Al Jazeera English', 'Associated Press',
    'Radio Dabanga', 'Belsat TV', 'NBC News',
    'Singapore', 'Khartoum', 'Niamey', 'Ashgabat',
    'Chairman', 'President', 'Members',
    'unity government', 'consensus decision making',
    'Sudanese Armed Forces',
    '2019 Draft Constitutional Declaration',
}

# Strings whose position field signals the whole record is noise
# (no meaningful country can be extracted, person is not a state leader)
POSITION_BLACKLIST = {
    'Patriotic Movement for Safeguard and Restoration',
    'National Committee of Reconciliation and Development',
    'High Military Command for the Restoration of National Security and Public Order',
    'National Council for the Safeguard of the Homeland',
    'Transitional Sovereignty Council',
    'Georgian Dream',
    'Polisario Front',
    'Houthis',
    'Libyan National Army',
    'Hamas government in Gaza',
    "Cambodian People's Party",
    "People's Council of Turkmenistan",
    "People's Council",
    'Radio Dabanga',
    'BBC News',
    'Al Jazeera',
    'Associated Press',
}
# Manual lookup for titles that carry no country info in their link
# e.g. 'Taoiseach' is the Irish PM title, 'Ngwenyama' is the Eswatini king title
TITLE_TO_COUNTRY = {
    'Taoiseach':          ('Prime Minister', 'Ireland'),
    'Ngwenyama':          ('King',           'Eswatini'),
    'Paramount Leader':   ('Paramount Leader', 'China'),
}

# Patterns in the article title that signal non-country content
NON_COUNTRY_PATTERNS = [
    r'^representatives\b',
    r'^co-princes\b',
    r'^members\b',
    r'^safeguard\b',
    r'^restoration\b',
    r'^reconciliation\b',
    r'^communist party\b',
    r"^workers'\s*party\b",
    r"^people's\s*(revolutionary\s*)?party\b",
    r'^presidential leadership\b',
    r'^re-foundation\b',
    r'^homeland\b',
    r'^security\b',
]

def clean(raw: str) -> str:
    """'Link|DisplayText' -> 'DisplayText'"""
    return raw.split('|')[-1].strip()


def get_link(raw: str) -> str:
    """'Link|DisplayText' -> 'Link'  (the Wikipedia article title)"""
    return raw.split('|')[0].strip()


def extract_country(position_raw: str) -> str | None:
    """
    Extract country name from the position field.
    Uses the Wikipedia article title (before '|') as the source,
    since it reliably contains the full country name.

    Examples:
      'President of France|President'                        -> 'France'
      'President of the Republic of China|President'         -> 'Republic of China'
      'President of the Democratic Republic of the Congo|...'-> 'Democratic Republic of the Congo'
      'High Representative for Bosnia and Herzegovina|...'   -> 'Bosnia and Herzegovina'
      'Paramount Leader'                                     -> None (handled via TITLE_TO_COUNTRY)
    """
    link = get_link(position_raw)

    # Match 'X of/for [the] COUNTRY' - handles both 'of' and 'for'
    m = re.search(r'\b(?:of|for) (?:the )?(.+)$', link, re.IGNORECASE)
    if not m:
        return None

    candidate = m.group(1).strip()
    # Remove leading 'the ' if present
    candidate = re.sub(r'^the\s+', '', candidate, flags=re.IGNORECASE).strip()

    # Filter out non-country phrases
    for pat in NON_COUNTRY_PATTERNS:
        if re.match(pat, candidate, re.IGNORECASE):
            return None

    return candidate if candidate else None


def extract_position(position_raw: str) -> str | None:
    """
    Extract short position title from the position field.

    Examples:
      'President of France|President'                               -> 'President'
      'General Secretary of the Communist Party of China|General Secretary' -> 'General Secretary'
      'Paramount Leader'                                            -> 'Paramount Leader'
      'Taoiseach'                                                   -> 'Taoiseach'
    """
    display = clean(position_raw)   # text after '|'
    link    = get_link(position_raw)

    # If '|' exists, the display text IS the short title
    if '|' in position_raw:
        return display if display else None

    # No '|': the whole string is the title
    # Remove 'of/for [the] ...' suffix to get short title
    short = re.sub(r'\s+(?:of|for)\s+.*$', '', link, flags=re.IGNORECASE).strip()
    return short if short else None


def is_noise_position(position_raw: str) -> bool:
    """Return True if the whole record should be skipped based on position field."""
    link    = get_link(position_raw)
    display = clean(position_raw)
    return (
        link    in POSITION_BLACKLIST or
        display in POSITION_BLACKLIST or
        any(link.startswith(p) for p in POSITION_BLACKLIST)
    )


def parse_record(record: list) -> list[dict]:
    """
    Parse one world_leaders record into a list of triples.
    Each triple: {'person': str, 'position': str, 'country': str}

    Rules:
      - record[0] is always the position field
      - record[1:] are person names (there can be multiple)
      - Skip the whole record if position is noise
      - Skip individual persons if they are in the blacklist
      - Drop triples where any of the three fields is empty/None
    """
    if len(record) < 2:
        return []

    position_raw = record[0]

    # Skip noisy records entirely
    if is_noise_position(position_raw):
        return []

    # Check for special title-only cases (no 'of' in link)
    link = get_link(position_raw)
    if link in TITLE_TO_COUNTRY:
        position, country = TITLE_TO_COUNTRY[link]
    else:
        position = extract_position(position_raw)
        country  = extract_country(position_raw)

    persons = [clean(r) for r in record[1:]]

    results = []
    for person in persons:
        if (
            person
            and person not in PERSON_BLACKLIST
            and position
            and country          # drop records with no country
        ):
            results.append({
                'person':   person,
                'position': position,
                'country':  country,
            })

    return results

In [53]:
# Deduplicate: same (person, position, country) should appear only once
seen = set()
triples = []
for record in world_leaders:
    for t in parse_record(record):
        key = (t['person'], t['position'], t['country'])
        if key not in seen:
            seen.add(key)
            triples.append(t)

print(f"Extracted {len(triples)} triples\n")
print(f"{'Person':<35} {'Position':<30} {'Country'}")
print("-" * 90)
for t in triples:
    print(f"{t['person']:<35} {t['position']:<30} {t['country']}")


Extracted 220 triples

Person                              Position                       Country
------------------------------------------------------------------------------------------
Hibatullah Akhundzada               Supreme Leader                 Afghanistan
Edi Rama                            Prime Minister                 Albania
Abdelmadjid Tebboune                President                      Algeria
Xavier Espot                        Prime Minister                 Andorra
João Lourenço                       President                      Angola
Rodney Williams                     Governor-General               Antigua and Barbuda
Gaston Browne                       Prime Minister                 Antigua and Barbuda
Javier Milei                        President                      Argentina
Nikol Pashinyan                     Prime Minister                 Armenia
Sam Mostyn                          Governor-General               Australia
Anthony Albanese              

In [54]:
# Build RDF graph
g = Graph()
WL = Namespace("http://worldleaders.org/ontology#")
WD = Namespace("http://worldleaders.org/data/")
g.bind("wl", WL)
g.bind("wd", WD)

g.add((WL.Person,         RDF.type, OWL.Class))
g.add((WL.Country,        RDF.type, OWL.Class))
g.add((WL.Position,       RDF.type, OWL.Class))
g.add((WL.leads,          RDF.type, OWL.ObjectProperty))
g.add((WL.holds_position, RDF.type, OWL.ObjectProperty))


def uri_safe(text: str) -> str:
    return re.sub(r'[^\w]', '_', text).strip('_')


for t in triples:
    p_uri   = WD[uri_safe(t['person'])]
    pos_uri = WD[uri_safe(t['position'])]
    c_uri   = WD[uri_safe(t['country'])]

    g.add((p_uri,   RDF.type,            WL.Person))
    g.add((p_uri,   RDFS.label,          Literal(t['person'],   lang="en")))
    g.add((pos_uri, RDF.type,            WL.Position))
    g.add((pos_uri, RDFS.label,          Literal(t['position'], lang="en")))
    g.add((c_uri,   RDF.type,            WL.Country))
    g.add((c_uri,   RDFS.label,          Literal(t['country'],  lang="en")))
    g.add((p_uri,   WL.holds_position,   pos_uri))
    g.add((p_uri,   WL.leads,            c_uri))

print(f"\nRDF graph: {len(g)} triples")
g.serialize(destination="world_leaders_rdf.ttl", format="turtle")
print("Saved: world_leaders_rdf.ttl")



RDF graph: 1319 triples
Saved: world_leaders_rdf.ttl
